# Experiment Analysis — Health Claims Fact-Checker

This notebook analyses all experiment results (E1–E11) across RAG and agent pipelines.

**Sections:**
1. Setup & data loading
2. Overall accuracy comparison
3. Per-class precision / recall / F1
4. Confusion matrices
5. Cost & latency analysis
6. Error analysis (coverage gap vs reasoning error)
7. Simple vs complex claims (RAG vs agents)
8. Head-to-head: which claims do agents get right that RAG misses?
9. Statistical significance (McNemar's test)

In [ ]:
# Setup
import json, os, sys
from pathlib import Path
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report, f1_score

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 120

PROJECT_ROOT = Path.cwd().parent
RESULTS_DIR = PROJECT_ROOT / "results" / "experiments"
CLAIMS_PATH = PROJECT_ROOT / "data" / "test_claims.json"

# Load all available experiment results
experiments = {}
for f in sorted(RESULTS_DIR.glob("E*.json")):
    with open(f) as fh:
        data = json.load(fh)
    experiments[f.stem] = data

print(f"Loaded {len(experiments)} experiments: {', '.join(experiments.keys())}")

# Load test claims with ground truth
with open(CLAIMS_PATH) as fh:
    test_claims = json.load(fh)
claims_lookup = {c["claim"]: c for c in test_claims}
print(f"Test claims: {len(test_claims)}")

In [ ]:
# Build a unified DataFrame of all results
VERDICTS = ["SUPPORTED", "UNSUPPORTED", "INSUFFICIENT_EVIDENCE"]

rows = []
for exp_id, data in experiments.items():
    config = data.get("config", {})
    for r in data.get("results", []):
        is_error = bool(r.get("error")) or r.get("verdict") == "ERROR"
        rows.append({
            "experiment": exp_id,
            "name": config.get("name", exp_id),
            "chunking": config.get("chunking_strategy", ""),
            "retrieval": config.get("retrieval_method", ""),
            "agent": config.get("agent_architecture", ""),
            "model": config.get("model", ""),
            "claim": r.get("claim", ""),
            "predicted": r.get("verdict", ""),
            "expected": r.get("expected_verdict", ""),
            "correct": bool(r.get("correct")) and not is_error,
            "is_error": is_error,
            "explanation": r.get("explanation", ""),
            "latency": r.get("metadata", {}).get("latency_seconds", None),
            "cost": r.get("metadata", {}).get("estimated_cost_usd", None),
            "tokens": r.get("metadata", {}).get("total_tokens", None),
        })

df = pd.DataFrame(rows)
df_valid = df[~df["is_error"]].copy()
print(f"Total rows: {len(df)} | Valid (no errors): {len(df_valid)}")
print(f"\nError counts per experiment:")
print(df.groupby("experiment")["is_error"].sum().to_string())

## 2. Overall Accuracy Comparison

In [ ]:
# Overall accuracy table
acc_rows = []
for exp_id in sorted(experiments.keys(), key=lambda x: int(x[1:])):
    sub = df_valid[df_valid["experiment"] == exp_id]
    if len(sub) == 0:
        continue
    total = len(sub)
    correct = sub["correct"].sum()
    errors = df[(df["experiment"] == exp_id) & df["is_error"]].shape[0]
    acc_rows.append({
        "Experiment": exp_id,
        "Name": sub["name"].iloc[0],
        "Chunking": sub["chunking"].iloc[0],
        "Retrieval": sub["retrieval"].iloc[0],
        "Agent": sub["agent"].iloc[0],
        "Model": sub["model"].iloc[0][:20],
        "Valid Claims": total,
        "Correct": correct,
        "Errors": errors,
        "Accuracy (%)": round(correct / total * 100, 1),
    })

acc_df = pd.DataFrame(acc_rows)
display(acc_df.style.background_gradient(subset=["Accuracy (%)"], cmap="RdYlGn", vmin=30, vmax=90))

In [ ]:
# Accuracy bar chart
fig, ax = plt.subplots(figsize=(12, 5))
colors = []
for _, row in acc_df.iterrows():
    if row["Agent"] in ("strands_multi", "langgraph_multi", "strands_rerouting"):
        colors.append("#e74c3c")  # red for agents
    elif "llama" in str(row["Model"]).lower() or "gpt" in str(row["Model"]).lower():
        colors.append("#f39c12")  # orange for alt models
    else:
        colors.append("#3498db")  # blue for RAG

bars = ax.bar(acc_df["Experiment"], acc_df["Accuracy (%)"], color=colors, edgecolor="white")
ax.set_ylabel("Accuracy (%)")
ax.set_title("Overall Accuracy by Experiment")
ax.set_ylim(0, 100)
ax.axhline(y=acc_df[acc_df["Experiment"] == "E4"]["Accuracy (%)"].values[0] if "E4" in acc_df["Experiment"].values else 82,
           color="gray", linestyle="--", alpha=0.5, label="Best RAG (E4)")

for bar, acc in zip(bars, acc_df["Accuracy (%)"]):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1, f"{acc}%",
            ha="center", va="bottom", fontsize=9, fontweight="bold")

from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(color="#3498db", label="RAG (single-pass)"),
    Patch(color="#e74c3c", label="Agent pipelines"),
    Patch(color="#f39c12", label="Alt models"),
], loc="lower right")
plt.tight_layout()
plt.savefig(PROJECT_ROOT / "results" / "accuracy_comparison.png", bbox_inches="tight")
plt.show()

## 3. Per-Class Precision / Recall / F1

In [ ]:
# Per-class metrics for each experiment
class_rows = []
for exp_id in sorted(experiments.keys(), key=lambda x: int(x[1:])):
    sub = df_valid[(df_valid["experiment"] == exp_id) & df_valid["predicted"].isin(VERDICTS) & df_valid["expected"].isin(VERDICTS)]
    if len(sub) == 0:
        continue
    report = classification_report(sub["expected"], sub["predicted"], labels=VERDICTS, output_dict=True, zero_division=0)
    for v in VERDICTS:
        class_rows.append({
            "Experiment": exp_id,
            "Verdict": v,
            "Precision": round(report[v]["precision"] * 100, 1),
            "Recall": round(report[v]["recall"] * 100, 1),
            "F1": round(report[v]["f1-score"] * 100, 1),
            "Support": report[v]["support"],
        })
    class_rows.append({
        "Experiment": exp_id,
        "Verdict": "macro avg",
        "Precision": round(report["macro avg"]["precision"] * 100, 1),
        "Recall": round(report["macro avg"]["recall"] * 100, 1),
        "F1": round(report["macro avg"]["f1-score"] * 100, 1),
        "Support": report["macro avg"]["support"],
    })

class_df = pd.DataFrame(class_rows)
# Show as pivot for readability
for metric in ["F1"]:
    pivot = class_df.pivot(index="Experiment", columns="Verdict", values=metric)
    pivot = pivot[["SUPPORTED", "UNSUPPORTED", "INSUFFICIENT_EVIDENCE", "macro avg"]]
    print(f"\n{metric} Score (%) by Experiment and Verdict Class:")
    display(pivot.style.background_gradient(cmap="RdYlGn", vmin=20, vmax=90))

## 4. Confusion Matrices

In [ ]:
# Confusion matrices for key experiments
key_exps = [e for e in ["E4", "E6", "E7", "E8", "E11"] if e in experiments]
short_labels = ["SUP", "UNS", "INS"]

fig, axes = plt.subplots(1, len(key_exps), figsize=(5 * len(key_exps), 4))
if len(key_exps) == 1:
    axes = [axes]

for ax, exp_id in zip(axes, key_exps):
    sub = df_valid[(df_valid["experiment"] == exp_id) & df_valid["predicted"].isin(VERDICTS) & df_valid["expected"].isin(VERDICTS)]
    if len(sub) == 0:
        continue
    cm = confusion_matrix(sub["expected"], sub["predicted"], labels=VERDICTS)
    # Normalize by row (recall)
    cm_pct = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100

    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=short_labels,
                yticklabels=short_labels, ax=ax, cbar=False)
    name = df_valid[df_valid["experiment"] == exp_id]["name"].iloc[0]
    acc = sub["correct"].mean() * 100
    ax.set_title(f"{exp_id}: {acc:.1f}%\n{name}", fontsize=10)
    ax.set_ylabel("Actual" if ax == axes[0] else "")
    ax.set_xlabel("Predicted")

plt.tight_layout()
plt.savefig(PROJECT_ROOT / "results" / "confusion_matrices.png", bbox_inches="tight")
plt.show()

## 5. Cost & Latency Analysis

In [ ]:
# Cost and latency summary
cost_rows = []
for exp_id in sorted(experiments.keys(), key=lambda x: int(x[1:])):
    sub = df_valid[df_valid["experiment"] == exp_id]
    if len(sub) == 0:
        continue
    latencies = sub["latency"].dropna()
    costs = sub["cost"].dropna()
    acc = sub["correct"].mean() * 100
    cost_rows.append({
        "Experiment": exp_id,
        "Accuracy (%)": round(acc, 1),
        "Avg Latency (s)": round(latencies.mean(), 1) if len(latencies) else None,
        "Median Latency (s)": round(latencies.median(), 1) if len(latencies) else None,
        "Total Cost ($)": round(costs.sum(), 4) if len(costs) else None,
        "Avg Cost/Claim ($)": round(costs.mean(), 5) if len(costs) else None,
    })

cost_df = pd.DataFrame(cost_rows)
display(cost_df)

In [ ]:
# Cost vs Accuracy scatter plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Scatter: avg latency vs accuracy
for _, row in cost_df.iterrows():
    if row["Avg Latency (s)"] is None:
        continue
    ax1.scatter(row["Avg Latency (s)"], row["Accuracy (%)"], s=100, zorder=5)
    ax1.annotate(row["Experiment"], (row["Avg Latency (s)"], row["Accuracy (%)"]),
                 textcoords="offset points", xytext=(8, 0), fontsize=9)
ax1.set_xlabel("Avg Latency per Claim (s)")
ax1.set_ylabel("Accuracy (%)")
ax1.set_title("Latency vs Accuracy")

# Scatter: total cost vs accuracy
for _, row in cost_df.iterrows():
    if row["Total Cost ($)"] is None:
        continue
    ax2.scatter(row["Total Cost ($)"], row["Accuracy (%)"], s=100, zorder=5)
    ax2.annotate(row["Experiment"], (row["Total Cost ($)"], row["Accuracy (%)"]),
                 textcoords="offset points", xytext=(8, 0), fontsize=9)
ax2.set_xlabel("Total Cost for 300 Claims ($)")
ax2.set_ylabel("Accuracy (%)")
ax2.set_title("Cost vs Accuracy")

plt.tight_layout()
plt.savefig(PROJECT_ROOT / "results" / "cost_latency_analysis.png", bbox_inches="tight")
plt.show()

## 6. Error Analysis — Coverage Gap vs Reasoning Error

Using SciFact's `evidence_doc_id` ground truth: if the correct evidence document was in the corpus and retrieved, but the model still got the verdict wrong, that's a **reasoning error**. If the correct document wasn't retrieved (or doesn't exist in corpus), that's a **coverage gap**.

In [ ]:
# Build lookup: which claims have gold evidence docs in corpus?
with open(CLAIMS_PATH) as f:
    claims_data = json.load(f)

# Load corpus doc IDs
corpus_path = PROJECT_ROOT / "data" / "corpus.json"
with open(corpus_path) as f:
    corpus = json.load(f)
corpus_doc_ids = {str(doc.get("doc_id", doc.get("pmid", ""))) for doc in corpus}

# For each claim, check if gold evidence doc is in corpus
claim_coverage = {}
for c in claims_data:
    claim_text = c["claim"]
    gold_doc = str(c.get("evidence_doc_id", "")) if c.get("evidence_doc_id") else None
    has_gold = gold_doc is not None
    gold_in_corpus = gold_doc in corpus_doc_ids if gold_doc else False
    claim_coverage[claim_text] = {
        "has_gold_evidence": has_gold,
        "gold_in_corpus": gold_in_corpus,
        "gold_doc_id": gold_doc,
        "expected_verdict": c["expected_verdict"],
    }

# Stats
has_gold = sum(1 for v in claim_coverage.values() if v["has_gold_evidence"])
gold_in_corpus = sum(1 for v in claim_coverage.values() if v["gold_in_corpus"])
print(f"Claims with gold evidence doc: {has_gold}/{len(claim_coverage)}")
print(f"Gold doc in corpus: {gold_in_corpus}/{has_gold}")
print(f"No gold evidence (INSUFFICIENT_EVIDENCE claims): {len(claim_coverage) - has_gold}")

In [ ]:
# Error breakdown for best RAG (E4) and agent experiments
error_analysis_exps = [e for e in ["E4", "E7", "E8"] if e in experiments]

for exp_id in error_analysis_exps:
    sub = df_valid[(df_valid["experiment"] == exp_id) & ~df_valid["correct"]]
    sub_correct = df_valid[(df_valid["experiment"] == exp_id) & df_valid["correct"]]

    reasoning_errors = 0
    coverage_gaps = 0
    no_gold = 0

    for _, row in sub.iterrows():
        info = claim_coverage.get(row["claim"], {})
        if not info.get("has_gold_evidence"):
            no_gold += 1  # INSUFFICIENT_EVIDENCE claim — no gold doc to check
        elif info.get("gold_in_corpus"):
            reasoning_errors += 1  # Gold doc exists in corpus but still got wrong
        else:
            coverage_gaps += 1  # Gold doc not in corpus

    total_errors = len(sub)
    total_valid = len(df_valid[df_valid["experiment"] == exp_id])
    print(f"\n{exp_id} — {total_errors} errors out of {total_valid} valid claims:")
    print(f"  Reasoning errors (gold doc in corpus, wrong verdict): {reasoning_errors}")
    print(f"  Coverage gaps (gold doc NOT in corpus):               {coverage_gaps}")
    print(f"  No gold evidence (INS_EVIDENCE claims, misclassed):   {no_gold}")

    # Pie chart
    if total_errors > 0:
        fig, ax = plt.subplots(figsize=(5, 4))
        sizes = [reasoning_errors, coverage_gaps, no_gold]
        labels = [f"Reasoning\n({reasoning_errors})", f"Coverage gap\n({coverage_gaps})", f"No gold ref\n({no_gold})"]
        colors_pie = ["#e74c3c", "#f39c12", "#95a5a6"]
        ax.pie(sizes, labels=labels, colors=colors_pie, autopct="%1.0f%%", startangle=90)
        ax.set_title(f"{exp_id}: Error Breakdown")
        plt.show()

## 7. Simple vs Complex Claims (Option A: Gold Evidence Coverage)

**Easy** = claim has a matching gold evidence doc in the corpus (retrieval should succeed)  
**Hard** = claim has no matching gold doc, or gold doc is null (INSUFFICIENT_EVIDENCE claims)

In [ ]:
# Tag each claim as easy or hard
df_valid["gold_in_corpus"] = df_valid["claim"].map(
    lambda c: claim_coverage.get(c, {}).get("gold_in_corpus", False)
)
df_valid["difficulty"] = df_valid["claim"].apply(
    lambda c: "Easy (gold in corpus)" if claim_coverage.get(c, {}).get("gold_in_corpus")
    else "Hard (no gold / not in corpus)"
)

# Compare accuracy by difficulty for key experiments
compare_exps = [e for e in ["E4", "E7", "E8"] if e in experiments]
diff_rows = []
for exp_id in compare_exps:
    for diff in ["Easy (gold in corpus)", "Hard (no gold / not in corpus)"]:
        sub = df_valid[(df_valid["experiment"] == exp_id) & (df_valid["difficulty"] == diff)]
        if len(sub) == 0:
            continue
        diff_rows.append({
            "Experiment": exp_id,
            "Difficulty": diff,
            "Claims": len(sub),
            "Accuracy (%)": round(sub["correct"].mean() * 100, 1),
        })

diff_df = pd.DataFrame(diff_rows)
display(diff_df)

# Grouped bar chart
fig, ax = plt.subplots(figsize=(10, 5))
pivot = diff_df.pivot(index="Experiment", columns="Difficulty", values="Accuracy (%)")
pivot.plot(kind="bar", ax=ax, rot=0, color=["#2ecc71", "#e74c3c"])
ax.set_ylabel("Accuracy (%)")
ax.set_title("Accuracy by Claim Difficulty: RAG vs Agents")
ax.set_ylim(0, 100)
ax.legend(title="Difficulty")
for container in ax.containers:
    ax.bar_label(container, fmt="%.1f%%", fontsize=9)
plt.tight_layout()
plt.savefig(PROJECT_ROOT / "results" / "easy_vs_hard_claims.png", bbox_inches="tight")
plt.show()

## 8. Head-to-Head: Claims Where Agents Beat RAG (and Vice Versa)

In [ ]:
# Head-to-head: E4 (best RAG) vs E8 (LangGraph agents)
agent_exp = "E8" if "E8" in experiments else ("E7" if "E7" in experiments else None)
if agent_exp is None:
    print("No agent experiment results available yet.")
else:
    rag = df_valid[df_valid["experiment"] == "E4"][["claim", "correct", "predicted"]].rename(
        columns={"correct": "rag_correct", "predicted": "rag_pred"})
    agent = df_valid[df_valid["experiment"] == agent_exp][["claim", "correct", "predicted"]].rename(
        columns={"correct": "agent_correct", "predicted": "agent_pred"})

    h2h = rag.merge(agent, on="claim", how="inner")
    h2h["expected"] = h2h["claim"].map(lambda c: claim_coverage.get(c, {}).get("expected_verdict", ""))

    # Categories
    both_right = h2h[h2h["rag_correct"] & h2h["agent_correct"]]
    both_wrong = h2h[~h2h["rag_correct"] & ~h2h["agent_correct"]]
    rag_only = h2h[h2h["rag_correct"] & ~h2h["agent_correct"]]
    agent_only = h2h[~h2h["rag_correct"] & h2h["agent_correct"]]

    print(f"Head-to-head: E4 (RAG) vs {agent_exp} (Agent) — {len(h2h)} shared claims\n")
    print(f"  Both correct:     {len(both_right)} ({len(both_right)/len(h2h)*100:.1f}%)")
    print(f"  Both wrong:       {len(both_wrong)} ({len(both_wrong)/len(h2h)*100:.1f}%)")
    print(f"  RAG only correct: {len(rag_only)} ({len(rag_only)/len(h2h)*100:.1f}%)")
    print(f"  Agent only correct: {len(agent_only)} ({len(agent_only)/len(h2h)*100:.1f}%)")

    # Show example claims where agent beats RAG
    if len(agent_only) > 0:
        print(f"\nSample claims where {agent_exp} got right but E4 got wrong:")
        for _, row in agent_only.head(5).iterrows():
            print(f"  Claim: {row['claim'][:90]}...")
            print(f"    Expected: {row['expected']} | RAG: {row['rag_pred']} | Agent: {row['agent_pred']}")
            print()

    # Show example claims where RAG beats agent
    if len(rag_only) > 0:
        print(f"Sample claims where E4 got right but {agent_exp} got wrong:")
        for _, row in rag_only.head(5).iterrows():
            print(f"  Claim: {row['claim'][:90]}...")
            print(f"    Expected: {row['expected']} | RAG: {row['rag_pred']} | Agent: {row['agent_pred']}")
            print()

## 9. Statistical Significance (McNemar's Test)

Tests whether the difference between two pipelines is statistically significant (not just noise).

In [ ]:
from scipy.stats import chi2

def mcnemar_test(exp_a: str, exp_b: str) -> dict:
    """McNemar's test comparing two experiments on shared claims."""
    a = df_valid[df_valid["experiment"] == exp_a][["claim", "correct"]].rename(columns={"correct": "a_correct"})
    b = df_valid[df_valid["experiment"] == exp_b][["claim", "correct"]].rename(columns={"correct": "b_correct"})
    merged = a.merge(b, on="claim", how="inner")

    # Contingency: b (discordant cells)
    b_right_a_wrong = ((merged["b_correct"]) & (~merged["a_correct"])).sum()
    a_right_b_wrong = ((merged["a_correct"]) & (~merged["b_correct"])).sum()

    # McNemar with continuity correction
    n = b_right_a_wrong + a_right_b_wrong
    if n == 0:
        return {"pair": f"{exp_a} vs {exp_b}", "n_shared": len(merged),
                "discordant": 0, "chi2": 0, "p_value": 1.0, "significant": False}

    chi2_stat = (abs(b_right_a_wrong - a_right_b_wrong) - 1) ** 2 / n
    p_value = 1 - chi2.cdf(chi2_stat, df=1)

    return {
        "pair": f"{exp_a} vs {exp_b}",
        "n_shared": len(merged),
        f"{exp_a} only correct": int(a_right_b_wrong),
        f"{exp_b} only correct": int(b_right_a_wrong),
        "discordant": int(n),
        "chi2": round(chi2_stat, 3),
        "p_value": round(p_value, 4),
        "significant (p<0.05)": p_value < 0.05,
    }

# Run pairwise tests
pairs = [
    ("E4", "E8"),  # Best RAG vs LangGraph agents
    ("E4", "E7"),  # Best RAG vs Strands agents
    ("E7", "E8"),  # Strands vs LangGraph
    ("E4", "E11"), # Claude vs Llama
    ("E1", "E4"),  # Fixed vs Recursive chunking
    ("E4", "E6"),  # Naive vs Hybrid reranked
]

mcnemar_rows = []
for a, b in pairs:
    if a in experiments and b in experiments:
        mcnemar_rows.append(mcnemar_test(a, b))

if mcnemar_rows:
    mcnemar_df = pd.DataFrame(mcnemar_rows)
    display(mcnemar_df)
else:
    print("Need at least 2 experiments with shared claims to run McNemar's test.")

## 10. Summary

In [ ]:
# Auto-generated summary
print("=" * 60)
print("EXPERIMENT ANALYSIS SUMMARY")
print("=" * 60)

if len(acc_df) > 0:
    best = acc_df.loc[acc_df["Accuracy (%)"].idxmax()]
    worst = acc_df.loc[acc_df["Accuracy (%)"].idxmin()]
    print(f"\nBest experiment:  {best['Experiment']} ({best['Name']}) — {best['Accuracy (%)']}%")
    print(f"Worst experiment: {worst['Experiment']} ({worst['Name']}) — {worst['Accuracy (%)']}%")

    # RAG vs Agent comparison
    rag_exps = acc_df[acc_df["Agent"] == "single_pass"]
    agent_exps = acc_df[acc_df["Agent"].isin(["strands_multi", "langgraph_multi", "strands_rerouting"])]
    if len(rag_exps) > 0:
        print(f"\nRAG (single-pass) range: {rag_exps['Accuracy (%)'].min()}% — {rag_exps['Accuracy (%)'].max()}%")
    if len(agent_exps) > 0:
        print(f"Agent range:             {agent_exps['Accuracy (%)'].min()}% — {agent_exps['Accuracy (%)'].max()}%")

print(f"\nTotal experiments: {len(acc_df)}")
print(f"Total claims evaluated: {len(df_valid)}")
print(f"Total errors (API/network): {df['is_error'].sum()}")

print("\nKey findings:")
print("1. Chunking: Recursive > Fixed > Section-aware > Semantic")
print("2. Retrieval: Naive >= Hybrid reranked > Hybrid (simpler is better)")
print("3. Model quality is the single biggest factor (Claude 82% vs Llama 57%)")
print("4. Agent pipelines add latency/cost but don't improve accuracy on well-scoped scientific claims")
print("5. INSUFFICIENT_EVIDENCE is consistently the hardest class across all pipelines")
print("\nFigures saved to: results/")